# Step 8: Pandas UDF & Apache Arrow

## Learning Objectives
1. Problems with Python UDFs (serialization overhead)
2. What is Apache Arrow?
3. Four types of Pandas UDF (Vectorized UDF)
4. Using mapInPandas / applyInPandas
5. Performance comparison: Python UDF vs Pandas UDF vs built-in functions
6. Real-world usage patterns

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time
import random

from datetime import datetime, timedelta
spark = SparkSession.builder \
    .appName('Step8-PandasUDF-Arrow') \
    .master('spark://spark-master:7077') \
    .config('spark.executor.memory', '1g') \
    .config('spark.sql.shuffle.partitions', '10') \
    .config('spark.sql.execution.arrow.pyspark.enabled', 'true') \
    .config('spark.sql.warehouse.dir', '/home/jovyan/data/warehouse') \
    .getOrCreate()

print(f'Arrow enabled: {spark.conf.get("spark.sql.execution.arrow.pyspark.enabled")}')
print(f'Spark version: {spark.version}')
print(f'✅ Spark UI: http://localhost:4040')

Arrow enabled: true
Spark version: 3.5.0
✅ Spark UI: http://localhost:4040


---
## 1. Problems with Python UDFs

```
Standard Python UDF execution flow:

  JVM (Spark Executor)          Python Worker
  ┌──────────────┐              ┌──────────────┐
  │ serialize row1│─── pickle ──→│ deserialize  │
  │              │              │ call function │
  │ deserialize  │←── pickle ──│ serialize    │
  │              │              │              │
  │ serialize row2│─── pickle ──→│ deserialize  │
  │  ...repeat...│              │  ...repeat...│
  └──────────────┘              └──────────────┘
       round-trip per row → slow!

Pandas UDF (Arrow-based):

  JVM (Spark Executor)          Python Worker
  ┌──────────────┐              ┌──────────────┐
  │ batch(1000s) │── Arrow ───→│  pd.Series    │
  │ columnar xfer│  zero-copy  │  vector ops   │
  │ receive result│←── Arrow ──│  return result│
  └──────────────┘              └──────────────┘
       batch + zero-copy → fast!
```

In [2]:
# Generate test data
random.seed(42)
n = 1_000_000

df = spark.range(n) \
    .withColumn('value', (F.rand() * 1000).cast('double')) \
    .withColumn('category', (F.col('id') % 10).cast('string')) \
    .withColumn('name', F.concat(F.lit('item_'), F.col('id').cast('string')))

df.cache().count()
print(f'Data: {n:,} rows')
df.show(5)

Data: 1,000,000 rows
+---+------------------+--------+------+
| id|             value|category|  name|
+---+------------------+--------+------+
|  0| 301.4248609144076|       0|item_0|
|  1| 485.4693372484594|       1|item_1|
|  2| 480.1145356893288|       2|item_2|
|  3|294.84714826279503|       3|item_3|
|  4|33.173830277834604|       4|item_4|
+---+------------------+--------+------+
only showing top 5 rows



In [3]:
# Standard Python UDF
from pyspark.sql.functions import udf
import math

@udf('double')
def python_udf_compute(value):
    """Row-by-row processing: slow"""
    if value is None:
        return None
    return math.log(value + 1) * math.sqrt(value) + math.sin(value)

# NOTE: we aggregate the result with .agg(F.sum(...)) instead of .count().
# .count() does NOT need the computed column, so Catalyst PRUNES the UDF and it
# never runs — making every "UDF benchmark" identical. Summing the result forces
# the UDF to execute on all rows, so the timing is real.
start = time.time()
df.select(python_udf_compute(F.col('value')).alias('result')).agg(F.sum('result')).collect()
python_udf_time = time.time() - start
print(f'Python UDF: {python_udf_time:.3f}s')

Python UDF: 0.906s


In [4]:
# Built-in function (optimal)
start = time.time()
df.select(
    (F.log(F.col('value') + 1) * F.sqrt(F.col('value')) + F.sin(F.col('value'))).alias('result')
).agg(F.sum('result')).collect()
builtin_time = time.time() - start
print(f'Built-in:   {builtin_time:.3f}s')
print(f'{python_udf_time/builtin_time:.1f}x faster than Python UDF')

Built-in:   0.138s
6.6x faster than Python UDF


---
## 2. Apache Arrow

**Arrow** is a columnar in-memory format that enables **zero-copy** data transfer between JVM and Python.

```
Row-based (pickle):        Columnar (Arrow):
┌─────┬─────┬─────┐       ┌─────────────┐
│ id  │ val │ cat │       │ id: [1,2,3] │  ← contiguous memory
├─────┼─────┼─────┤       │ val:[.1,.2] │  ← contiguous memory
│  1  │ 0.1 │  A  │       │ cat:[A,B,C] │  ← contiguous memory
│  2  │ 0.2 │  B  │       └─────────────┘
│  3  │ 0.3 │  C  │       Optimal for SIMD/vector ops
└─────┴─────┴─────┘       zero-copy compatible with pandas
must serialize each row
```

### Enable Arrow
```python
spark.conf.set('spark.sql.execution.arrow.pyspark.enabled', 'true')
```

In [5]:
# toPandas performance comparison with Arrow
small_df = df.limit(100_000)

# Arrow disabled
spark.conf.set('spark.sql.execution.arrow.pyspark.enabled', 'false')
start = time.time()
pdf1 = small_df.toPandas()
no_arrow_time = time.time() - start

# Arrow enabled
spark.conf.set('spark.sql.execution.arrow.pyspark.enabled', 'true')
start = time.time()
pdf2 = small_df.toPandas()
arrow_time = time.time() - start

print(f'toPandas() without Arrow: {no_arrow_time:.3f}s')
print(f'toPandas() with Arrow:    {arrow_time:.3f}s')
print(f'Arrow is {no_arrow_time/arrow_time:.1f}x faster')
print(f'\n💡 Arrow accelerates both toPandas() and createDataFrame(pdf).')

toPandas() without Arrow: 0.392s
toPandas() with Arrow:    0.285s
Arrow is 1.4x faster

💡 Arrow accelerates both toPandas() and createDataFrame(pdf).


---
## 3. Four Types of Pandas UDF

| Type | Input | Output | Use case |
|------|-------|--------|----------|
| **Series → Series** | pd.Series | pd.Series | 1:1 column transformation |
| **Iterator[Series] → Iterator[Series]** | iterator | iterator | Large data + expensive initialization |
| **Multiple Series → Series** | multiple pd.Series | pd.Series | Multi-column input |
| **Series → Scalar** | pd.Series | scalar | Group-level aggregation |

In [6]:
import pandas as pd
import numpy as np
from pyspark.sql.functions import pandas_udf, PandasUDFType

# === Type 1: Series → Series ===
# The most basic form: transforms a single column.

@pandas_udf('double')
def pandas_compute(value: pd.Series) -> pd.Series:
    """Vectorized operation: fast"""
    return np.log(value + 1) * np.sqrt(value) + np.sin(value)

# Again: aggregate the result to force execution (not .count(), which prunes it).
start = time.time()
df.select(pandas_compute(F.col('value')).alias('result')).agg(F.sum('result')).collect()
pandas_udf_time = time.time() - start

print('=== Series → Series ===')
print(f'Python UDF:  {python_udf_time:.3f}s')
print(f'Pandas UDF:  {pandas_udf_time:.3f}s')
print(f'Built-in:    {builtin_time:.3f}s')
print(f'\nPandas UDF is {python_udf_time/pandas_udf_time:.1f}x faster than Python UDF')

=== Series → Series ===
Python UDF:  0.906s
Pandas UDF:  0.356s
Built-in:    0.138s

Pandas UDF is 2.5x faster than Python UDF


In [7]:
# === Type 2: Iterator[Series] → Iterator[Series] ===
# Useful when initialization (e.g., model loading) is expensive (done only once)

from typing import Iterator

@pandas_udf('double')
def pandas_iter_compute(batches: Iterator[pd.Series]) -> Iterator[pd.Series]:
    # Perform expensive initialization (e.g., model load) only once here
    # model = load_model('path/to/model')  # example
    
    for batch in batches:
        yield np.log(batch + 1) * np.sqrt(batch) + np.sin(batch)

start = time.time()
df.withColumn('result', pandas_iter_compute(F.col('value'))).count()
iter_udf_time = time.time() - start

print('=== Iterator[Series] → Iterator[Series] ===')
print(f'Elapsed: {iter_udf_time:.3f}s')
print()
print('💡 Use scenarios:')
print('   - ML model inference (load model once → predict per batch)')
print('   - DB connection pool (connect once → query per batch)')
print('   - External API calls (create session once → request per batch)')

=== Iterator[Series] → Iterator[Series] ===
Elapsed: 0.069s

💡 Use scenarios:
   - ML model inference (load model once → predict per batch)
   - DB connection pool (connect once → query per batch)
   - External API calls (create session once → request per batch)


In [8]:
# === Type 3: Multiple Series → Series ===
# Takes multiple columns as input and produces one output column

@pandas_udf('double')
def weighted_score(value: pd.Series, category: pd.Series) -> pd.Series:
    """Score with per-category weights"""
    weights = {'0': 1.0, '1': 1.5, '2': 0.8, '3': 2.0, '4': 1.2,
               '5': 0.9, '6': 1.1, '7': 1.3, '8': 0.7, '9': 1.8}
    w = category.map(weights).fillna(1.0)
    return value * w

start = time.time()
result = df.withColumn('score', weighted_score(F.col('value'), F.col('category')))
result.count()
multi_time = time.time() - start

print('=== Multiple Series → Series ===')
print(f'Elapsed: {multi_time:.3f}s')
result.select('id', 'value', 'category', 'score').show(5)

=== Multiple Series → Series ===
Elapsed: 0.068s
+---+------------------+--------+------------------+
| id|             value|category|             score|
+---+------------------+--------+------------------+
|  0| 301.4248609144076|       0| 301.4248609144076|
|  1| 485.4693372484594|       1| 728.2040058726891|
|  2| 480.1145356893288|       2| 384.0916285514631|
|  3|294.84714826279503|       3| 589.6942965255901|
|  4|33.173830277834604|       4|39.808596333401525|
+---+------------------+--------+------------------+
only showing top 5 rows



In [9]:
# === Type 4: Series → Scalar (aggregation function) ===
# Used with groupBy for group-level aggregation
# Note: Pandas UDF aggregates cannot be mixed with built-in aggregates in the same agg()

@pandas_udf('double')
def pandas_median(v: pd.Series) -> float:
    '''Median (not available as a Spark built-in)'''
    return float(v.median())

@pandas_udf('double')
def pandas_iqr(v: pd.Series) -> float:
    '''IQR (Interquartile Range)'''
    return float(v.quantile(0.75) - v.quantile(0.25))

start = time.time()

# Built-in aggregation
builtin_agg = df.groupBy('category').agg(
    F.avg('value').alias('mean'),
    F.stddev('value').alias('stddev'),
)

# Pandas UDF aggregation (run separately)
pandas_agg = df.groupBy('category').agg(
    pandas_median('value').alias('median'),
    pandas_iqr('value').alias('iqr'),
)

# Join
agg_result = builtin_agg.join(pandas_agg, 'category').orderBy('category')
agg_result.show()
scalar_time = time.time() - start

print(f'Elapsed: {scalar_time:.3f}s')
print()
print('💡 Statistical functions not available in Spark (median, IQR, etc.)')
print('   can be implemented efficiently as Pandas UDFs.')
print()
print('⚠️ Pandas UDF aggregates and built-in aggregates cannot be mixed in the same agg().')
print('   Compute them separately and then join.')


+--------+------------------+------------------+------------------+------------------+
|category|              mean|            stddev|            median|               iqr|
+--------+------------------+------------------+------------------+------------------+
|       0| 499.8999093430319|288.75288769138814| 500.3736563202352| 500.0074099524719|
|       1|501.48387779124783| 288.7491728568569| 503.5708566650447| 499.9957118706911|
|       2|498.29114509770415| 288.6716115776748|497.48613947099443|499.99197516364063|
|       3|500.78422417452157|288.65759563099164| 501.8695675993705| 499.6500440049702|
|       4| 498.5509316911509|288.24793531950974|  497.439160311552| 499.6885211964237|
|       5| 501.0860295007023| 288.8375333630635| 502.0557098501124|501.00305602514015|
|       6| 500.7051688425814| 288.3347115919104| 501.4803665874315|497.85951150104574|
|       7|498.82798477177226| 288.2917858145926|   498.68730956312| 497.6592751516736|
|       8|500.81749702225403|  288.13616908

---
## 4. mapInPandas & applyInPandas

| Function | Use | Unit |
|----------|-----|------|
| **mapInPandas** | Process the entire DataFrame via pandas | Partition |
| **applyInPandas** | Process each group via pandas after groupBy | Group |

In [10]:
# === mapInPandas ===
# Receive a pandas DataFrame per partition and process it

def normalize_partition(iterator: Iterator[pd.DataFrame]) -> Iterator[pd.DataFrame]:
    """Min-max normalize the value column within each partition"""
    for pdf in iterator:
        v_min = pdf['value'].min()
        v_max = pdf['value'].max()
        v_range = v_max - v_min
        if v_range > 0:
            pdf['normalized'] = (pdf['value'] - v_min) / v_range
        else:
            pdf['normalized'] = 0.0
        yield pdf[['id', 'value', 'category', 'normalized']]

result_schema = StructType([
    StructField('id', LongType()),
    StructField('value', DoubleType()),
    StructField('category', StringType()),
    StructField('normalized', DoubleType()),
])

start = time.time()
normalized = df.mapInPandas(normalize_partition, schema=result_schema)
normalized.count()
map_time = time.time() - start

print(f'mapInPandas: {map_time:.3f}s')
normalized.show(5)

print('💡 mapInPandas processes data partition by partition.')
print('   Note: normalization is within a partition, not across the full dataset!')

mapInPandas: 0.280s
+---+------------------+--------+-------------------+
| id|             value|category|         normalized|
+---+------------------+--------+-------------------+
|  0| 301.4248609144076|       0|0.30134493283798863|
|  1| 485.4693372484594|       1|0.48543412977165395|
|  2| 480.1145356893288|       2|0.48007802705999053|
|  3|294.84714826279503|       3| 0.2947656218811201|
|  4|33.173830277834604|       4|0.03302872041940319|
+---+------------------+--------+-------------------+
only showing top 5 rows

💡 mapInPandas processes data partition by partition.
   Note: normalization is within a partition, not across the full dataset!


In [11]:
# === applyInPandas ===
# Receive a pandas DataFrame per group after groupBy
# → enables complex pandas operations within each group

def group_stats(pdf: pd.DataFrame) -> pd.DataFrame:
    """Per-group statistics + Z-score calculation"""
    mean = pdf['value'].mean()
    std = pdf['value'].std()
    
    pdf['z_score'] = (pdf['value'] - mean) / std if std > 0 else 0
    pdf['group_mean'] = mean
    pdf['group_std'] = std
    pdf['is_outlier'] = (pdf['z_score'].abs() > 2.0)
    
    return pdf[['id', 'value', 'category', 'z_score', 'group_mean', 'group_std', 'is_outlier']]

stats_schema = StructType([
    StructField('id', LongType()),
    StructField('value', DoubleType()),
    StructField('category', StringType()),
    StructField('z_score', DoubleType()),
    StructField('group_mean', DoubleType()),
    StructField('group_std', DoubleType()),
    StructField('is_outlier', BooleanType()),
])

start = time.time()
stats = df.groupBy('category').applyInPandas(group_stats, schema=stats_schema)
stats.count()
apply_time = time.time() - start

print(f'applyInPandas: {apply_time:.3f}s')
stats.filter(F.col('is_outlier') == True).show(10)

outlier_count = stats.filter(F.col('is_outlier') == True).count()
total = stats.count()
print(f'Outliers: {outlier_count:,} / {total:,} ({outlier_count/total*100:.1f}%)')

applyInPandas: 0.758s
+---+-----+--------+-------+----------+---------+----------+
| id|value|category|z_score|group_mean|group_std|is_outlier|
+---+-----+--------+-------+----------+---------+----------+
+---+-----+--------+-------+----------+---------+----------+

Outliers: 0 / 1,000,000 (0.0%)


In [12]:
# applyInPandas real-world example: per-group linear regression

def group_linear_fit(pdf: pd.DataFrame) -> pd.DataFrame:
    """Per-group linear regression (value ~ id)"""
    x = pdf['id'].values.astype(float)
    y = pdf['value'].values
    
    # least-squares fit with numpy
    n = len(x)
    if n < 2:
        return pd.DataFrame({
            'category': pdf['category'].iloc[:1],
            'slope': [0.0], 'intercept': [0.0],
            'r_squared': [0.0], 'n_points': [n]
        })
    
    x_mean, y_mean = x.mean(), y.mean()
    ss_xy = ((x - x_mean) * (y - y_mean)).sum()
    ss_xx = ((x - x_mean) ** 2).sum()
    
    slope = ss_xy / ss_xx if ss_xx > 0 else 0
    intercept = y_mean - slope * x_mean
    
    y_pred = slope * x + intercept
    ss_res = ((y - y_pred) ** 2).sum()
    ss_tot = ((y - y_mean) ** 2).sum()
    r_sq = 1 - ss_res / ss_tot if ss_tot > 0 else 0
    
    return pd.DataFrame({
        'category': [pdf['category'].iloc[0]],
        'slope': [slope],
        'intercept': [intercept],
        'r_squared': [r_sq],
        'n_points': [n]
    })

fit_schema = StructType([
    StructField('category', StringType()),
    StructField('slope', DoubleType()),
    StructField('intercept', DoubleType()),
    StructField('r_squared', DoubleType()),
    StructField('n_points', IntegerType()),
])

start = time.time()
fits = df.groupBy('category').applyInPandas(group_linear_fit, schema=fit_schema)
fits.orderBy('category').show()
fit_time = time.time() - start

print(f'Per-group linear regression: {fit_time:.3f}s')
print('\n💡 applyInPandas enables per-group ML model training and inference.')
print('   scikit-learn, statsmodels, etc. can be used as-is.')

+--------+--------------------+------------------+--------------------+--------+
|category|               slope|         intercept|           r_squared|n_points|
+--------+--------------------+------------------+--------------------+--------+
|       0|1.058024186148088...|499.37090254007785|1.118823593104778...|  100000|
|       1|2.723831266590426...| 500.1219730532745|7.415526663501204E-6|  100000|
|       2|3.822868625304137E-6| 496.3797222536584|1.461482738873698...|  100000|
|       3|-1.75689679083528...| 501.6626690561428|3.087092312670236...|  100000|
|       4|-8.16413620035479...| 498.9591376847573|6.685150198970646E-7|  100000|
|       5|-9.87017400200544E-7| 501.5795382008016|9.731178967076204E-7|  100000|
|       6|-2.19080352326384...| 500.8147092378256|4.811008236238479E-8|  100000|
|       7|-3.16768696210067...|  498.986369753412|1.006104459744250...|  100000|
|       8|-2.84583187886309E-6| 502.2404214991784|8.129166550241429E-6|  100000|
|       9|-1.61807054062109.

---
## 5. Comprehensive Performance Comparison

In [13]:
# Compare the same operation using multiple approaches
# Operation: log(x+1) * sqrt(x) + sin(x) on the value column
#
# KEY: each approach ends with .agg(F.sum('r')).collect(), NOT .count().
# .count() lets Catalyst prune the computed column so the UDF never runs — which makes
# all methods look identical (and can even make Python UDF "win" by pure noise).
# Summing the result forces every row through the function, giving real timings.

def time_expr(make_col, n_runs=3):
    times = []
    for _ in range(n_runs):
        start = time.time()
        df.select(make_col().alias('r')).agg(F.sum('r')).collect()
        times.append(time.time() - start)
    return sum(times) / n_runs

results = {
    'Built-in':          time_expr(lambda: F.log(F.col('value')+1) * F.sqrt('value') + F.sin('value')),
    'Pandas UDF':        time_expr(lambda: pandas_compute(F.col('value'))),
    'Pandas UDF (Iter)': time_expr(lambda: pandas_iter_compute(F.col('value'))),
    'Python UDF':        time_expr(lambda: python_udf_compute(F.col('value'))),
}

# Print results
print(f'=== Performance Comparison ({n:,} rows, avg of 3 runs, result materialized) ===')
print(f'{"Method":<20} {"Time":>8} {"Ratio":>8}')
print('-' * 40)
baseline = results['Built-in']
for name, elapsed in sorted(results.items(), key=lambda x: x[1]):
    ratio = elapsed / baseline
    bar = '█' * int(ratio * 10)
    print(f'{name:<20} {elapsed:>7.3f}s {ratio:>7.1f}x  {bar}')

print(f'''
💡 Performance order: Built-in > Pandas UDF ≈ Pandas UDF(Iter) >> Python UDF

   Decision guide:
   1. Achievable with built-in functions → Built-in (always best)
   2. Need vectorized computation → Pandas UDF (Series)
   3. Expensive initialization (model load, etc.) → Pandas UDF (Iterator)
   4. Complex per-group processing → applyInPandas
   5. Last resort → Python UDF

   ⚠️ Benchmark lesson: end with .agg()/write, never .count(), or Catalyst prunes the
      computed column and the UDF never executes — the timings become meaningless.
''')

=== Performance Comparison (1,000,000 rows, avg of 3 runs, result materialized) ===
Method                   Time    Ratio
----------------------------------------
Built-in               0.066s     1.0x  ██████████
Pandas UDF (Iter)      0.132s     2.0x  ████████████████████
Pandas UDF             0.151s     2.3x  ██████████████████████
Python UDF             0.412s     6.2x  ██████████████████████████████████████████████████████████████

💡 Performance order: Built-in > Pandas UDF ≈ Pandas UDF(Iter) >> Python UDF

   Decision guide:
   1. Achievable with built-in functions → Built-in (always best)
   2. Need vectorized computation → Pandas UDF (Series)
   3. Expensive initialization (model load, etc.) → Pandas UDF (Iterator)
   4. Complex per-group processing → applyInPandas
   5. Last resort → Python UDF

   ⚠️ Benchmark lesson: end with .agg()/write, never .count(), or Catalyst prunes the
      computed column and the UDF never executes — the timings become meaningless.



---
## 6. Real-World Usage Patterns

In [14]:
# Pattern 1: String processing (regex + text)
import re

@pandas_udf('string')
def clean_text(texts: pd.Series) -> pd.Series:
    """Text cleaning: remove HTML tags and special characters"""
    def clean(t):
        if pd.isna(t):
            return None
        t = re.sub(r'<[^>]+>', '', t)         # remove HTML tags
        t = re.sub(r'[^a-zA-Z0-9\s]', '', t)  # remove special characters
        t = re.sub(r'\s+', ' ', t).strip()    # normalize whitespace
        return t
    return texts.apply(clean)

text_df = spark.createDataFrame([
    ('<p>Hello! <b>Spark</b> learning in progress.</p>',),
    ('<div>Price: \\$100 (discount applied)</div>',),
    ('   lots   of   extra    spaces   in   this   text   ',),
    (None,),
], ['raw_text'])

text_df.withColumn('cleaned', clean_text('raw_text')).show(truncate=False)

+----------------------------------------------------+---------------------------------+
|raw_text                                            |cleaned                          |
+----------------------------------------------------+---------------------------------+
|<p>Hello! <b>Spark</b> learning in progress.</p>    |Hello Spark learning in progress |
|<div>Price: \$100 (discount applied)</div>          |Price 100 discount applied       |
|   lots   of   extra    spaces   in   this   text   |lots of extra spaces in this text|
|NULL                                                |NULL                             |
+----------------------------------------------------+---------------------------------+



In [15]:
# Pattern 2: Time-series processing (per-group moving averages, cumulative sum)

# Generate time-series data
ts_data = []
for cat in ['A', 'B', 'C']:
    base = random.uniform(100, 500)
    for day in range(30):
        ts_data.append((
            cat,
            (datetime(2025, 6, 1) + timedelta(days=day)).strftime('%Y-%m-%d'),
            round(base + random.gauss(0, 20), 2)
        ))

ts_df = spark.createDataFrame(ts_data, ['category', 'date', 'value'])

def add_ts_features(pdf: pd.DataFrame) -> pd.DataFrame:
    """Add time-series features per group"""
    pdf = pdf.sort_values('date')
    pdf['ma_7'] = pdf['value'].rolling(7, min_periods=1).mean()
    pdf['ma_14'] = pdf['value'].rolling(14, min_periods=1).mean()
    pdf['cumsum'] = pdf['value'].cumsum()
    pdf['pct_change'] = pdf['value'].pct_change().fillna(0)
    pdf['volatility_7'] = pdf['value'].rolling(7, min_periods=1).std().fillna(0)
    return pdf

ts_schema = StructType([
    StructField('category', StringType()),
    StructField('date', StringType()),
    StructField('value', DoubleType()),
    StructField('ma_7', DoubleType()),
    StructField('ma_14', DoubleType()),
    StructField('cumsum', DoubleType()),
    StructField('pct_change', DoubleType()),
    StructField('volatility_7', DoubleType()),
])

ts_result = ts_df.groupBy('category').applyInPandas(add_ts_features, schema=ts_schema)

print('=== Time-series features (category=A) ===')
ts_result.filter(F.col('category') == 'A').orderBy('date').show(10)

=== Time-series features (category=A) ===
+--------+----------+------+------------------+------------------+------------------+--------------------+------------------+
|category|      date| value|              ma_7|             ma_14|            cumsum|          pct_change|      volatility_7|
+--------+----------+------+------------------+------------------+------------------+--------------------+------------------+
|       A|2025-06-01|371.61|            371.61|            371.61|            371.61|                 0.0|               0.0|
|       A|2025-06-02|358.28|           364.945|           364.945|            729.89|-0.03587093996394075| 9.425733393216708|
|       A|2025-06-03|361.24|363.71000000000004|363.71000000000004|           1091.13| 0.00826169476387184| 6.999849998392837|
|       A|2025-06-04|387.97|           369.775|           369.775|1479.1000000000001|  0.0739951278928137|13.409033025042017|
|       A|2025-06-05|337.01|363.22200000000004|363.22200000000004|1816.11000

In [16]:
# Pattern 3: ML model inference (Iterator pattern)

# Simple model simulation (production would use sklearn/pytorch models)
class SimpleModel:
    def __init__(self):
        # Production: pickle.load() or torch.load(), etc.
        self.coefficients = np.array([0.3, -0.1, 0.5])
        self.intercept = 10.0
    
    def predict(self, features: np.ndarray) -> np.ndarray:
        return features @ self.coefficients + self.intercept

@pandas_udf('double')
def model_predict(iterator: Iterator[tuple[pd.Series, pd.Series, pd.Series]]) -> Iterator[pd.Series]:
    # Load model once per partition
    model = SimpleModel()
    
    for feat1, feat2, feat3 in iterator:
        features = np.column_stack([feat1.values, feat2.values, feat3.values])
        predictions = model.predict(features)
        yield pd.Series(predictions)

# Feature data
pred_df = spark.range(100_000) \
    .withColumn('feat1', F.rand() * 10) \
    .withColumn('feat2', F.rand() * 5) \
    .withColumn('feat3', F.rand() * 20)

start = time.time()
predictions = pred_df.withColumn(
    'prediction', 
    model_predict(F.col('feat1'), F.col('feat2'), F.col('feat3'))
)
predictions.count()
pred_time = time.time() - start

print(f'Model inference (100K rows): {pred_time:.3f}s')
predictions.select('feat1', 'feat2', 'feat3', 'prediction').show(5)

print('\n💡 The Iterator pattern loads the model only once per partition.')
print('   GPU models can use the same pattern.')

Model inference (100K rows): 0.073s
+-------------------+------------------+------------------+------------------+
|              feat1|             feat2|             feat3|        prediction|
+-------------------+------------------+------------------+------------------+
| 4.8793751032798705| 2.655316780132995| 2.251332269087547|12.323946987514436|
|  7.753125534566245|3.5259553384176416|6.8071771563186045|15.376930704687412|
|  5.734847098403656|0.7196384577000559| 8.359344058357985|15.828162312930084|
|0.41550920683060877|  2.56338424364597|18.573616402056356|19.155122538712764|
|0.15955953467768347|2.0940354078847707|3.8928138973844084|11.784871268307032|
+-------------------+------------------+------------------+------------------+
only showing top 5 rows


💡 The Iterator pattern loads the model only once per partition.
   GPU models can use the same pattern.


---
## 📝 Key Takeaways

| Concept | Description |
|---------|-------------|
| **Apache Arrow** | Columnar in-memory format, zero-copy between JVM and Python |
| **Python UDF** | Row-by-row pickle serialization → slow |
| **Pandas UDF** | Batch Arrow transfer + numpy vectorized ops → fast |
| **Series → Series** | 1:1 column transformation (basic) |
| **Iterator** | Expensive initialization (model loading) |
| **Series → Scalar** | Group-level aggregation (median, IQR, etc.) |
| **mapInPandas** | Partition-level pandas processing |
| **applyInPandas** | Per-group pandas processing (ML, time-series) |

### Decision Flow
```
Achievable with built-ins? → Yes → Built-in function
                              No
                              ↓
Vectorized operation? → Yes → Pandas UDF (Series)
                         No
                         ↓
Expensive init? → Yes → Pandas UDF (Iterator)
                   No
                   ↓
Per-group logic? → Yes → applyInPandas
                    No
                    ↓
                Python UDF (last resort)
```

### Next Step (Step 9)
- Spark internal architecture
- DAG Scheduler and Task Scheduler
- Stage/Task splitting principles
- BlockManager and data flow

In [17]:
spark.stop()
print('SparkSession stopped')

SparkSession stopped
